In [2]:
import sys

print(sys.version)
print(sys.executable)


import torch

print(torch.__version__)
print(torch.backends.mps.is_available())  # True

device = torch.device("mps")

3.12.16 (main, Oct 26 2026, 13:59:40) [Clang 23.1.3 ]
/Users/probee/programming/llm_from_scratch/.venv/bin/python
2.15.0
True


# Loading tokenizer


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

text = "Hello, how are you?"

tokens = tokenizer.encode(text)

for token in tokens:
    print(f"{token} ->> {tokenizer.decode([token])}")

print(f"vocab size: {tokenizer.vocab_size}")

9707 ->> Hello
11 ->> ,
1246 ->>  how
525 ->>  are
498 ->>  you
30 ->> ?
vocab size: 151643


# Loding base model


In [4]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

model_id = "Qwen/Qwen3-0.6B-Base"

path = hf_hub_download(repo_id=model_id, filename="model.safetensors")

weights = load_file(path, device="cpu")

print(list(weights.keys())[:10])

['model.embed_tokens.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.0.self_attn.k_norm.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.q_norm.weight']


In [5]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

model.eval()

W1114 16:25:13.474000 3434 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [6]:
print(model)  # Complete architecture
print(model.model.layers[0])  # First transformer block
weights = model.state_dict()  # All pretrained tensors
config = model.config  # Architecture hyperparameters

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

# Inference


In [ ]:
@torch.inference_mode()  # no backprop; dont track gradient
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    model.eval()  # enables deterministic behavior

    for _ in range(max_new_tokens):
        # forward  pass: (batch, seq_len. vocab size)
        logits = model(token_ids).logits

        # gen last token's logits and select higesht-scoring token
        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

        # yield each token as soon as it generated. so you are streaming token
        yield next_token

        # append newly predicted token to the sequence
        token_ids = torch.cat([token_ids, next_token], dim=1)

## KV caching


In [ ]:
@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
):
    model.eval()

    # Process the entire prompt once (prefill)
    outputs = model(token_ids, use_cache=True)
    cache = outputs.past_key_values
    logits = outputs.logits[:, -1, :]

    for _ in range(max_new_tokens):
        # Select the highest-scoring next token
        next_token = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

        yield next_token

        # Process ONLY the new token, reusing cached K/V tensors
        outputs = model(
            next_token,
            past_key_values=cache,
            use_cache=True,
        )

        cache = outputs.past_key_values
        logits = outputs.logits[:, -1, :]

In [14]:
prompt = "Explain large language models in a single sentence."

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt), device=device
).unsqueeze(0)

max_new_tokens = 100

In [ ]:
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
):
    # with flush= True ,Immediately sends printed text to the output stream rather than waiting for buffering.
    print(tokenizer.decode(token[0]), end="", flush=True)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. Large language models are designed to mimic this complexity and adapt to new contexts and languages, making them powerful tools for a wide range of applications, from customer service to content creation.

In [17]:
for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
):
    print(tokenizer.decode(token[0]), end="", flush=True)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and punctuation marks, which are used to convey meaning and express thoughts and ideas. The structure of human

## torch.compile


In [18]:
compiled_model = torch.compile(
    model,
    backend="inductor",
    mode="default",
)

In [19]:
with torch.inference_mode():
    logits = compiled_model(
        input_token_ids_tensor,
        use_cache=False,
    ).logits

print(logits.shape)

torch.Size([1, 10, 151936])


In [ ]:
import time


def benchmark(model, n=100):
    # Warmup
    with torch.inference_mode():
        for _ in range(3):
            model(input_token_ids_tensor, use_cache=False)

    torch.mps.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        for _ in range(n):
            model(input_token_ids_tensor, use_cache=False)

    torch.mps.synchronize()

    return (time.perf_counter() - start) / n


eager_time = benchmark(model)
compiled_time = benchmark(compiled_model)

print(f"Eager:    {eager_time * 1000:.2f} ms")
print(f"Compiled: {compiled_time * 1000:.2f} ms")
print(f"Speedup:  {eager_time / compiled_time:.2f}x")

Eager:    44.99 ms
Compiled: 39.97 ms
Speedup:  1.13x
